In [6]:
import numpy as np
import pennylane as qml

In [7]:
theta = 5 / 8
n_estimation = 3

target_wire = 0
estimation_wires = list(range(1, n_estimation + 1))

U = np.diag([1, np.exp(2j * np.pi * theta)])

In [10]:
dev = qml.device("default.qubit", wires=n_estimation + 1)

@qml.qnode(dev)
def circuit():
    # Prepare eigenstate
    qml.PauliX(wires=target_wire)

    # Call phase estimation method
    qml.QuantumPhaseEstimation(
        U,
        target_wires=target_wire,
        estimation_wires=estimation_wires,
    )

    return qml.probs(wires=estimation_wires)

In [11]:
probabilities = circuit()
result = np.argmax(probabilities)
estimated_phase = result / (2 ** n_estimation)

print("Measured bits:", format(result, f"0{n_estimation}b"))
print("Estimated phase:", estimated_phase)
print("Peak probability:", probabilities[result])

Measured bits: 101
Estimated phase: 0.625
Peak probability: 1.0


In [12]:
# Manual estimation
def my_phase_estimation(U, target_wire, estimation_wires):
    for wire in estimation_wires:
        qml.Hadamard(wires=wire)
    
    for j, wire in enumerate(estimation_wires):
        power = 2 ** (n_estimation - 1 - j)
        U_power = np.linalg.matrix_power(U, power)

        qml.ctrl(qml.QubitUnitary, control=wire)(
            U_power, wires=[target_wire]
        )

    qml.adjoint(qml.QFT)(wires=estimation_wires)

In [13]:
# Manual estimation 
man_dev = qml.device("default.qubit", wires=n_estimation + 1)

@qml.qnode(man_dev)
def manual_circuit():
    # Prepare eigenstate
    qml.PauliX(wires=target_wire)

    # Call manual phase estimation method
    my_phase_estimation(U, target_wire, estimation_wires)

    return qml.probs(wires=estimation_wires)

In [14]:
probabilities = manual_circuit()
result = np.argmax(probabilities)
estimated_phase = result / (2 ** n_estimation)

print("Measured bits:", format(result, f"0{n_estimation}b"))
print("Estimated phase:", estimated_phase)
print("Peak probability:", probabilities[result])

Measured bits: 101
Estimated phase: 0.625
Peak probability: 1.0
